# Sprint 11 · Modelado de Datos y Dashboards con Power BI: Caso Megaline 📊

**Tema:** Modelado de datos, medidas avanzadas y dashboards interactivos en Power BI.

> 🚀 En esta sesión, transformaremos datos crudos de **Megaline** en un dashboard interactivo y revelador. Aprenderás a construir un modelo de datos robusto, escribirás tus primeras fórmulas en DAX y diseñarás un panel que comunica ideas de negocio de forma efectiva.

**Programa:** Data Analytics · **Sprint:** 11 · **Modalidad:** Práctico-Teórico

## 🎯 Objetivos de la sesión

Al finalizar esta clase, serás capaz de:

1.  Conectar múltiples fuentes de datos en Power BI.
2.  Construir un **modelo en estrella** para asegurar la integridad de tu análisis.
3.  Crear **columnas calculadas y medidas** usando el lenguaje DAX.
4.  Aplicar la función `CALCULATE` para modificar el contexto de tus cálculos.
5.  Diseñar un **dashboard interactivo** que responda a preguntas de negocio clave.

## 📈 El Desafío: Analizando los Datos de Megaline

Megaline, una empresa de telecomunicaciones, quiere entender mejor el comportamiento de sus clientes. Necesitan saber qué plan es más rentable y cómo se distribuye el consumo de llamadas, mensajes y datos. ¡Tu misión es ayudarlos a visualizar esta información!

### Nuestras Fuentes de Datos

Trabajaremos con 5 tablas que describen el negocio:

| Tabla | Descripción | Rol en el modelo |
|---|---|---|
| **users** | Información demográfica de los clientes. | Dimensión |
| **plans** | Detalles de las tarifas `Surf` y `Ultimate`. | Dimensión |
| **calls** | Registro de todas las llamadas realizadas. | Tabla de Hechos |
| **messages**| Registro de todos los SMS enviados. | Tabla de Hechos |
| **internet**| Registro del consumo de datos en cada sesión. | Tabla de Hechos |

> 🤔 **Reflexión:** ¿Por qué tenemos tres tablas de hechos? Porque los eventos (llamadas, mensajes, datos) ocurren de forma independiente y tienen distinta granularidad. Nuestro modelo deberá reflejar esto.

> Nota: Megaline redondea los segundos a minutos y los megabytes a gigabytes. Para las llamadas, cada llamada individual se redondea: incluso si la llamada duró solo un segundo, se contará como un minuto. Para el tráfico web, las sesiones web individuales no se redondean. En vez de esto, el total del mes se redondea hacia arriba. Si alguien usa 1025 megabytes este mes, se le cobrarán 2 gigabytes.

In [5]:
import pandas as pd
import numpy as np


In [3]:
megaline_calls = pd.read_csv('https://practicum-content.s3.us-west-1.amazonaws.com/datasets/megaline_calls.csv')
megaline_internet = pd.read_csv('https://practicum-content.s3.us-west-1.amazonaws.com/datasets/megaline_internet.csv')
megaline_messages = pd.read_csv('https://practicum-content.s3.us-west-1.amazonaws.com/datasets/megaline_messages.csv')
megaline_plans = pd.read_csv('https://practicum-content.s3.us-west-1.amazonaws.com/datasets/megaline_plans.csv')
megaline_users = pd.read_csv('https://practicum-content.s3.us-west-1.amazonaws.com/datasets/megaline_users.csv')


In [11]:
megaline_internet.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104825 entries, 0 to 104824
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            104825 non-null  object 
 1   user_id       104825 non-null  int64  
 2   session_date  104825 non-null  object 
 3   mb_used       104825 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 3.2+ MB


In [20]:
megaline_calls_agg=(
    megaline_calls
        .assign(
            call_date = lambda df: pd.to_datetime(df.call_date,errors='coerce'),
            year_month = lambda df: df.call_date.dt.strftime("%y-%m"),
            call_duration= lambda df: np.ceil(df.duration)
        )
        .groupby(["year_month","user_id"],as_index=False)
        .agg(
            min_usage = ('call_duration','sum')
        )
)
megaline_messages_agg=(
    megaline_messages
        .assign(
            message_date = lambda df: pd.to_datetime(df.message_date,errors='coerce'),
            year_month = lambda df: df.message_date.dt.strftime("%y-%m")
        )
        .groupby(["year_month","user_id"],as_index=False)
        .agg(
            sms_usage = ('id','nunique')
        )
)

megaline_messages_agg

megaline_internet_agg=(
    megaline_internet
        .assign(
            session_date = lambda df: pd.to_datetime(df.session_date,errors='coerce'),
            year_month = lambda df: df.session_date.dt.strftime("%y-%m")
        )
        .groupby(["year_month","user_id"],as_index=False)
        .agg(
            mb_usage = ('mb_used','sum')
        )
        .assign(
            gb_used = lambda df: np.ceil(df.mb_usage/1024)
        )
        .drop(columns=['mb_usage'])
)

users_with_usage_by_month = pd.concat([megaline_calls_agg[['user_id','year_month']],megaline_messages_agg[['user_id','year_month']],megaline_internet_agg[['user_id','year_month']]],axis=0).drop_duplicates()


megaline_summary_usage = (
        users_with_usage_by_month
        .merge(megaline_calls_agg, how='left', on=['user_id','year_month'])
        .merge(megaline_internet_agg, how='left', on=['user_id','year_month'])
        .merge(megaline_messages_agg, how='left', on=['user_id','year_month'])
        .merge(megaline_users[['user_id','plan']],how='left', on=['user_id'])
        .fillna({'gb_used':0,'sms_usage':0,'min_usage':0})
)

In [21]:
megaline_summary_usage.query('user_id == 1482').sort_values(by=['year_month'])

,user_id,year_month,min_usage,gb_used,sms_usage,plan
2275,1482,18-10,0.0,0.0,2.0,ultimate
1784,1482,18-11,785.0,20.0,87.0,ultimate
2241,1482,18-12,56.0,1.0,5.0,ultimate


## 🛠️ Paso 1: Construyendo el Modelo en Estrella

Un buen dashboard empieza con un buen modelo. Un **modelo en estrella** es como el chasis de un coche: le da estructura y solidez a todo lo que construyas encima.

**Tu Misión:**

1.  **Conectar los datos:** En Power BI, usa `Obtener datos` para cargar las 5 tablas de Megaline (pueden ser CSV o de una base de datos).
2.  **Ir a la Vista de Modelo:** Aquí es donde ocurre la magia.
3.  **Crear relaciones:** Power BI podría crear algunas relaciones automáticamente. ¡Verifícalas! Arrastra y suelta las columnas para crear las siguientes conexiones:

    *   `users[user_id]` → `calls[user_id]` (Uno a Varios)
    *   `users[user_id]` → `messages[user_id]` (Uno a Varios)
    *   `users[user_id]` → `internet[user_id]` (Uno a Varios)
    *   `users[plan]` → `plans[plan_name]` (Uno a Varios)

El esquema debería verse así, con `users` y `plans` como dimensiones centrales que describen los hechos.

```
         ┌─────────┐
         │  plans  │
         └────┬────┘
              │ plan_name
┌────────┐    │
│  users │────┼────> (calls, messages, internet)
└────────┘ user_id   (Tablas de Hechos)
```

> ✅ **Tip:** Asegúrate de que la dirección del filtro sea siempre **desde la dimensión hacia la tabla de hechos** (la flecha apunta hacia la tabla de hechos).

## 🧠 Paso 2: ¡Hablando el Idioma de los Datos con DAX!

**DAX (Data Analysis Expressions)** es el lenguaje de fórmulas de Power BI. Al principio puede parecer intimidante, pero es pura lógica.

> **Regla de Oro:** Prefiere siempre **Medidas** sobre **Columnas Calculadas**. Las medidas son dinámicas y se calculan según los filtros de tu visual, ¡lo que las hace súper potentes y eficientes!

### 2.1 El Prerrequisito: La Tabla Calendario

Para analizar datos por mes, trimestre o año, necesitas una tabla calendario. Esta es la columna vertebral de todo análisis temporal.

**Tu Misión:**
1.  Ve a la `Vista de datos`.
2.  En la cinta `Herramientas de tablas`, haz clic en `Nueva tabla`.
3.  Pega el siguiente código DAX para crear una tabla calendario dinámica:

In [ ]:
Calendar = 
VAR Fechas =
    UNION (
        SELECTCOLUMNS ( megaline_calls, "Fecha", megaline_calls[call_date] ),
        SELECTCOLUMNS ( megaline_internet, "Fecha", megaline_internet[session_date] ),
        SELECTCOLUMNS ( megaline_messages, "Fecha", megaline_messages[message_date] )
    )
RETURN
ADDCOLUMNS (
    CALENDAR (
        MINX ( Fechas, [Fecha] ),
        MAXX ( Fechas, [Fecha] )
    ),
    "Year", YEAR ( [Date] ),
    "YearMonth", FORMAT ( [Date], "YYYY-MM" ),
    "IdMonth", MONTH ( [Date] )
)

4.  **Importante:** Ve a la `Vista de modelo` y crea una relación entre `Calendar[YearMonth]` y `internet[session_date]`, `calls[call_date]`, y `messages[message_date]`.
5.  Finalmente, haz clic derecho en la tabla `Calendar` y selecciona `Marcar como tabla de fechas`.

### 2.2 El Gran Desafío: Calcular la Factura Mensual por Usuario

Aquí es donde demostramos el poder de DAX. Tu pregunta es clave: ¿cómo calculamos la factura de cada usuario, cada mes, basándonos en su consumo y su plan? Lo haremos con medidas, no creando tablas físicas.

#### Paso A: Medidas de Consumo Básico

Primero, las medidas simples que agregan el total.

In [ ]:
// 1. Total de minutos de llamada
Total Minutos = 
SUMX (
    megaline_calls,
    CEILING ( megaline_calls[duration], 1 )
)

// 2. Total de mensajes enviados
Total SMS = COUNT(messages[id])

// 3. Total de datos consumidos (en GB)
Total GB Usados = 
CEILING ( SUM ( megaline_internet[mb_used] ) / 1024, 1 )

Ahora integremos en un tabla que agrupe los consumos mensuales por usuarios

In [ ]:
fact_usage_monthly = 
ADDCOLUMNS(
    CROSSJOIN(
        VALUES(megaline_users[user_id]),
        VALUES(Calendar[YearMonth])
    ),
    "minutos", COALESCE(
        CALCULATE(
            SUMX(
                megaline_calls,
                CEILING(megaline_calls[duration], 1)
            )
        ),
        0
    ),
    "sms", COALESCE(
        CALCULATE(COUNT(megaline_messages[id])),
        0
    ),
    "gb", COALESCE(
        CALCULATE(
            CEILING(SUM(megaline_internet[mb_used]) / 1024, 1)
        ),
        0
    ),
    "plan", CALCULATE(SELECTEDVALUE(megaline_users[plan]))
)

Para los cálculos tambien necesitaremos obtener los datos de cada plan

In [ ]:
fact_usage_with_plan = 
ADDCOLUMNS(
    fact_usage_monthly,

    "renta_base",
    LOOKUPVALUE(
        megaline_plans[usd_monthly_pay],
        megaline_plans[plan_name], fact_usage_monthly[plan]
    ),

    "city",
    LOOKUPVALUE(
        megaline_users[city],
        megaline_users[user_id], fact_usage_monthly[user_id]
    ),

    "minutos_incluidos",
    LOOKUPVALUE(
        megaline_plans[minutes_included],
        megaline_plans[plan_name], fact_usage_monthly[plan]
    ),

    "sms_incluidos",
    LOOKUPVALUE(
        megaline_plans[messages_included],
        megaline_plans[plan_name], fact_usage_monthly[plan]
    ),

    "gb_incluidos",
    DIVIDE(
        LOOKUPVALUE(
            megaline_plans[mb_per_month_included],
            megaline_plans[plan_name], fact_usage_monthly[plan]
        ),
        1024,
        0
    ),

    "costo_minuto_extra",
    LOOKUPVALUE(
        megaline_plans[usd_per_minute],
        megaline_plans[plan_name], fact_usage_monthly[plan]
    ),

    "costo_sms_extra",
    LOOKUPVALUE(
        megaline_plans[usd_per_message],
        megaline_plans[plan_name], fact_usage_monthly[plan]
    ),

    "costo_gb_extra",
    LOOKUPVALUE(
        megaline_plans[usd_per_gb],
        megaline_plans[plan_name], fact_usage_monthly[plan]
    )
)

> El crossjoin puede generar relaciones artificiales entre el mes y los usuarios, se te queda como reta agregar la funcionalidad de solo considerar meses que sean mayores o iguales la fecha del registro del cliente y que no corresponda posterior a su fecha de cancelación si es que tuviera

#### Paso B: Medidas para Costos Adicionales

Ahora, creamos la lógica para calcular el costo de cada excedente. Estas medidas funcionarán dinámicamente para cualquier usuario/mes seleccionado.

In [ ]:
fact_consumption_analysis = 
ADDCOLUMNS(
    fact_usage_with_plan,

    //----------------------------------
    // CONSUMO EXTRA
    //----------------------------------

    "minutos_extra",
    MAX(
        0,
        fact_usage_with_plan[minutos] - fact_usage_with_plan[minutos_incluidos]
    ),

    "sms_extra",
    MAX(
        0,
        fact_usage_with_plan[sms] - fact_usage_with_plan[sms_incluidos]
    ),

    "gb_extra",
    MAX(
        0,
        fact_usage_with_plan[gb] - fact_usage_with_plan[gb_incluidos]
    ),

    //----------------------------------
    // MONTO POR SERVICIO (EXTRAS)
    //----------------------------------

    "monto_minutos_extra",
    MAX(
        0,
        fact_usage_with_plan[minutos] - fact_usage_with_plan[minutos_incluidos]
    ) * fact_usage_with_plan[costo_minuto_extra],

    "monto_sms_extra",
    MAX(
        0,
        fact_usage_with_plan[sms] - fact_usage_with_plan[sms_incluidos]
    ) * fact_usage_with_plan[costo_sms_extra],

    "monto_gb_extra",
    MAX(
        0,
        fact_usage_with_plan[gb] - fact_usage_with_plan[gb_incluidos]
    ) * fact_usage_with_plan[costo_gb_extra],

    //----------------------------------
    // TOTAL EXTRA
    //----------------------------------

    "monto_total_extra",
    MAX(
        0,
        fact_usage_with_plan[minutos] - fact_usage_with_plan[minutos_incluidos]
    ) * fact_usage_with_plan[costo_minuto_extra]
    +
    MAX(
        0,
        fact_usage_with_plan[sms] - fact_usage_with_plan[sms_incluidos]
    ) * fact_usage_with_plan[costo_sms_extra]
    +
    MAX(
        0,
        fact_usage_with_plan[gb] - fact_usage_with_plan[gb_incluidos]
    ) * fact_usage_with_plan[costo_gb_extra],

    //----------------------------------
    // RENTA BASE
    //----------------------------------

    "renta_mensual",
    fact_usage_with_plan[renta_base] +
    MAX(
        0,
        fact_usage_with_plan[minutos] - fact_usage_with_plan[minutos_incluidos]
    ) * fact_usage_with_plan[costo_minuto_extra]
    +
    MAX(
        0,
        fact_usage_with_plan[sms] - fact_usage_with_plan[sms_incluidos]
    ) * fact_usage_with_plan[costo_sms_extra]
    +
    MAX(
        0,
        fact_usage_with_plan[gb] - fact_usage_with_plan[gb_incluidos]
    ) * fact_usage_with_plan[costo_gb_extra]
    ,

    //----------------------------------
    // RENTA BASE
    //----------------------------------

    "renta_mensual_base",
    fact_usage_with_plan[renta_base],


    //----------------------------------
    // INDICADORES
    //----------------------------------

    "tuvo_consumo",
    IF(
        fact_usage_with_plan[minutos] > 0
            || fact_usage_with_plan[sms] > 0
            || fact_usage_with_plan[gb] > 0,
        1,
        0
    ),

    "tuvo_sobreconsumo",
    IF(
        MAX(0, fact_usage_with_plan[minutos] - fact_usage_with_plan[minutos_incluidos]) > 0
            || MAX(0, fact_usage_with_plan[sms] - fact_usage_with_plan[sms_incluidos]) > 0
            || MAX(0, fact_usage_with_plan[gb] - fact_usage_with_plan[gb_incluidos]) > 0,
        1,
        0
    )
)

## 🎨 Paso 3: Creando un Dashboard para la Gerencia

Un buen dashboard cuenta una historia. La gerencia de Megaline no quiere ver tablas y números, quiere respuestas.

**Preguntas a Responder:**

*   ¿Cuál es el ingreso total por plan? 
*   ¿Qué plan tiene más usuarios?
*   ¿Cómo es el consumo promedio (minutos, SMS, datos) por usuario para cada plan?
*   ¿Hay usuarios que exceden consistentemente los límites de su plan? (¡Tus medidas de costo extra responden esto!)

**Tu Misión:**

1.  **KPIs Principales:** Usa tarjetas (`Card`) para mostrar las métricas más importantes:
    *   `[Facturacion Total]`
    *   Número de Clientes (puedes crear una medida `COUNTROWS(users)`)
    *   `[Costo GB Extra]` (para ver cuánto se gana por datos adicionales)

2.  **Gráficos de Comparación:**
    *   **Gráfico de barras agrupadas:** Compara `[Facturacion Total]` por `plan`.
    *   **Gráfico de barras apiladas:** Muestra la distribución de usuarios por `ciudad` y `plan`.

3.  **Visual de Tendencia:**
    *   **Gráfico de líneas:** Muestra la evolución de `[Facturacion Total]` a lo largo del tiempo, usando tu `Calendario[Mes]`.

4.  **Añade Interactividad:**
    *   Usa un **Segmentador de datos (Slicer)** para filtrar todo el dashboard por `plan` o `ciudad`. ¡Verás cómo todos los gráficos se actualizan al instante!

> ✨ **Toque final:** Usa colores consistentes. Por ejemplo, asigna siempre el mismo color al plan `Surf` y otro al plan `Ultimate` en todos los gráficos. Esto hace que tu dashboard sea más fácil de leer y entender.

In [ ]:
Usuarios Activos =
VAR UsuariosConActividad =
    DISTINCT (
        UNION (
            SELECTCOLUMNS ( megaline_calls, "user_id", megaline_calls[user_id] ),
            SELECTCOLUMNS ( megaline_messages, "user_id", megaline_messages[user_id] ),
            SELECTCOLUMNS ( megaline_internet, "user_id", megaline_internet[user_id] )
        )
    )
RETURN
    COUNTROWS ( UsuariosConActividad )

## 🎉 Cierre y Próximos Pasos

¡Felicidades! Has pasado de datos crudos a un dashboard funcional y con propósito. Y no solo eso, has construido un motor de cálculo de facturación complejo y robusto.

**Hoy aprendiste a:**

*   Estructurar un modelo de datos en estrella y la importancia de una **tabla calendario**.
*   Escribir fórmulas DAX para manejar lógica de negocio compleja (cálculo de excedentes).
*   Usar **iteradores como `SUMX`** para realizar cálculos a nivel de fila dentro de una medida.
*   Diseñar un dashboard que responde preguntas de negocio profundas.

